# Attribute-Based Emergence Analysis  
### Using Layerwise Linear Probes on ResNet-50  
**Dataset:** CUB-200-2011  
**Goal:** Understand when different semantic properties become decodable inside a CNN

---

## Why This Notebook Exists
Many interpretability papers assume a “human-like” hierarchical pipeline:

> **Colors & textures → parts → objects → fine-grained species**

But this is rarely tested.

This notebook:
- Loads precomputed linear probing results
- Computes _emergence depth_ for each target
- Analyzes semantic groups of attributes
- Compares species vs attributes
- Produces figures suitable for posters/reports

---

## What We Will Measure

We define:

**Emergence depth**:  
> The earliest layer ℓ where performance ≥ 0.9 × max accuracy for that target.

This compresses an accuracy-vs-depth curve into one meaningful number.

Example:  
If for one attribute the best accuracy is 0.88 at layer4,  
and layer2 already achieves 0.80 (≥ 90% of 0.88),  
then emergence depth = layer2.

---

### Expected Patterns (Hypothesis)
| Semantic Type | Expected Emergence |
|---|---|
| Colors | Very early (conv1/layer1) |
| Simple shapes (bill length, wing shape) | Mid-depth |
| Location-specific (breast-pattern vs back-pattern) | Later |
| Species | Latest |

This notebook will evaluate whether these assumptions hold.


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

In [ ]:
# CELL 1 — Imports and setup

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 30)

# constants
RESULT_PATH = Path("/scratch/network/cr7998/cv_emergence_project/results/probes/resnet50_cub_probes_attributes.json")

LAYERS_ORDERED = [
    "conv1",
    "layer1",
    "layer2",
    "layer3",
    "layer4",
    "avgpool",
]


# Step 1: Load Probe Results
We trained **linear probes** to decode:

- Species identity (200-way classification)
- Binary attribute presence (≈200 valid attributes)

For each:
- We saved accuracy per layer
- For each layer–target pair, we directly record the best validation accuracy

Now we load that file and examine its structure.


In [ ]:
# CELL 2 — Load JSON results correctly

import json

assert RESULT_PATH.exists(), f"{RESULT_PATH} not found"

with open(RESULT_PATH, "r") as f:
    data = json.load(f)  # returns list of dicts

df = pd.DataFrame(data)

print("Loaded", df.shape[0], "rows")
df.head()


## What is in this table?

One row ≈ one `(layer, target)` combination:

Example columns:

| layer   | target_type | target_name | best_val_acc | group             |
|---------|-------------|-------------|--------------|------------------|
| layer3  | attribute   | has_wing_color::brown | 0.77 | has_wing_color |

This resembles a structured benchmark matrix.

Next, we compute the **emergence depth**.


In [ ]:
# CELL 3 — Compute emergence depth

# First: compute best accuracy per target
best_acc = df.groupby("target_name")["best_val_acc"].max().rename("max_acc")
df = df.merge(best_acc, on="target_name")

# threshold for emergence: >= 90% of max performance
df["threshold"] = 0.9 * df["max_acc"]

# indicator if layer exceeds threshold
df["hits_threshold"] = df["val_acc"] >= df["threshold"]

# emergence depth = first such layer in ordered list
emergence_records = []
for target in df["target_name"].unique():
    sub = df[df["target_name"] == target]
    for layer in LAYERS_ORDERED:
        row = sub[sub["layer"] == layer]
        if len(row) > 0 and row["hits_threshold"].iloc[0]:
            emergence_records.append({
                "target_name": target,
                "group": row["group"].iloc[0],
                "max_acc": row["max_acc"].iloc[0],
                "emergence_layer": layer
            })
            break

emerge_df = pd.DataFrame(emergence_records)
print("Total targets measured:", len(emerge_df))
emerge_df.head()


# Step 2: Evaluate Distribution of Emergence Depth

We group by attribute type (e.g., "has_bill_color", "has_wing_shape")  
to see where different semantics emerge.


In [ ]:
# CELL 4 — Summary table by group (robust version)

# 1. Count how many targets per group
group_counts = (
    emerge_df.groupby("group")
    .size()
    .rename("num_attributes")
    .sort_values(ascending=False)
)

print("Number of groups:", group_counts.shape[0])
print("Top 15 groups by attribute count:")
display(group_counts.head(15))

# 2. Compute the most common emergence layer per group
group_mode_layer = (
    emerge_df.groupby("group")["emergence_layer"]
    .agg(lambda x: x.value_counts().index[0])  # most frequent emergence layer
    .rename("most_common_layer")
)

# 3. Combine counts + mode layer into a single summary table
summary = pd.concat([group_mode_layer, group_counts], axis=1).reset_index()

print("\nFull summary (unfiltered):")
display(summary.head(20))

# 4. OPTIONAL: apply a *looser* filter, e.g. ≥ 3 attributes
MIN_ATTRS = 3   # change to 5 if you want stricter, 1 if you want everything

summary_filtered = summary[summary["num_attributes"] >= MIN_ATTRS]

print(f"\nFiltered summary (groups with ≥ {MIN_ATTRS} attributes): "
      f"{summary_filtered.shape[0]} groups remain")

summary_filtered.sort_values(by="num_attributes", ascending=False).head(20)


In [ ]:
summary_filtered = summary[summary["num_attributes"] >= 10]
summary_filtered.sort_values(by="most_common_layer").head(20)


## Interpreting this Table

In [ ]:
# CELL 5 — Map layer → numeric index for plotting

layer_to_idx = {layer: i for i, layer in enumerate(LAYERS_ORDERED)}
emerge_df["layer_idx"] = emerge_df["emergence_layer"].map(layer_to_idx)

# species-only depth
species_depth = emerge_df[emerge_df["group"] == "species"]["layer_idx"].iloc[0]
species_depth


## Plot: Group-wise Emergence Layer

The bar chart below shows mean emergence depth grouped by semantic category.


In [ ]:
# CELL 6 — Barplot of mean emergent depth

agg = emerge_df.groupby("group")["layer_idx"].mean().sort_values()

plt.figure(figsize=(10, 14))
sns.barplot(x=agg.values, y=agg.index, palette="viridis")
plt.title("Mean Emergence Depth per Attribute Group (lower = earlier)")
plt.xlabel("Layer depth index (0=conv1)")
plt.tight_layout()
plt.show()


# Step 3: Contrast Species vs Attributes

This answers:

**Does species emerge later than attributes?**

— If yes → model likely uses attributes → intuitive  
— If no → model may rely on global texture/background cues


In [ ]:
# CELL 7 — Compare species vs attribute emergence

species_row = emerge_df[emerge_df["group"]=="species"].iloc[0]
species_layer_idx = species_row["layer_idx"]

mean_attr = emerge_df[emerge_df["group"]!="species"]["layer_idx"].mean()

print("Species emergence depth index:", species_layer_idx)
print("Mean attribute emergence depth index:", mean_attr)

print("\nDifference:", species_layer_idx - mean_attr)


In [ ]:
# CELL 8 — Distribution histogram

plt.figure(figsize=(8,6))
sns.histplot(emerge_df["layer_idx"], bins=len(LAYERS_ORDERED), kde=False)
plt.xticks(range(len(LAYERS_ORDERED)), LAYERS_ORDERED)
plt.xlabel("Emergence layer")
plt.ylabel("Number of targets")
plt.title("Emergence Distribution Across All Attributes")
plt.show()


# Step 5: Layerwise Concept Saturation Curve
This shows:
- _How many attributes have already emerged_ at each depth  
- Equivalent to cumulative concept availability


In [ ]:
# CELL 9 — Cumulative emergence curve

layer_counts = emerge_df["layer_idx"].value_counts().sort_index()

cum_counts = layer_counts.cumsum()
plt.figure(figsize=(10,5))
plt.plot(cum_counts.index, cum_counts.values, marker='o')
plt.xticks(range(len(LAYERS_ORDERED)), LAYERS_ORDERED)
plt.ylabel("# attributes emerged")
plt.xlabel("Layer")
plt.title("Cumulative Attribute Emergence")
plt.grid()
plt.show()


In [ ]:
# CELL — Inspect retained vs discarded attributes

from datasets.cub_metadata import load_cub_metadata
from pathlib import Path

# Load original attribute metadata
meta = load_cub_metadata(Path("/scratch/network/cr7998/cv_emergence_project/data/CUB_200_2011/"))

attr_binary = meta.image_attributes_binary.set_index("image_id")
attr_cols = [c for c in attr_binary.columns if c.startswith("attr_")]

# Compute frequency per attribute
freqs = attr_binary[attr_cols].mean()

# Apply same filtering thresholds
low_threshold = 0.10
high_threshold = 0.90

discarded_too_rare = freqs[freqs < low_threshold]
discarded_too_common = freqs[freqs > high_threshold]
retained = freqs[(freqs >= low_threshold) & (freqs <= high_threshold)]

print("===== DISCARDED – too rare (<10%) =====")
display(pd.DataFrame(discarded_too_rare, columns=["frequency"]))

print("\n===== DISCARDED – too common (>90%) =====")
display(pd.DataFrame(discarded_too_common, columns=["frequency"]))

print("\n===== RETAINED USED ATTRIBUTES =====")
display(pd.DataFrame(retained, columns=["frequency"]))


In [ ]:
retained.size


In [ ]:
# ============================================
# NOTEBOOK CELL — Composite plots for report
# ============================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# 0) Load probe results
# -------------------------
# Update these if your filenames differ
BASELINE_JSON = Path("/scratch/network/cr7998/cv_emergence_project/results/probes/resnet50_cub_probes_attributes.json")
CBM_JSON      = Path("/scratch/network/cr7998/cv_emergence_project/results/probes/resnet50_cub_cbm_probes_attributes.json")

def load_probe_json(path: Path, model_name: str) -> pd.DataFrame:
    assert path.exists(), f"Missing: {path}"
    with open(path, "r") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    df["model"] = model_name
    return df

# If df_all already exists, we reuse it; otherwise load from disk.
if "df_all" in globals() and isinstance(df_all, pd.DataFrame):
    df = df_all.copy()
else:
    df = pd.concat([
        load_probe_json(BASELINE_JSON, "baseline"),
        load_probe_json(CBM_JSON, "cbm"),
    ], ignore_index=True)

print("Loaded df:", df.shape)
print("Models:", df["model"].unique())
print("Columns:", df.columns.tolist())

# -------------------------
# 1) Canonical layer order
# -------------------------
LAYER_ORDER = ["conv1", "layer1", "layer2", "layer3", "layer4", "avgpool"]
LAYER_TO_IDX = {l:i for i,l in enumerate(LAYER_ORDER)}

def df_with_layer_idx(dfin: pd.DataFrame) -> pd.DataFrame:
    out = dfin.copy()
    out["layer"] = pd.Categorical(out["layer"], categories=LAYER_ORDER, ordered=True)
    out["layer_idx"] = out["layer"].map(LAYER_TO_IDX)
    return out

df = df_with_layer_idx(df)

# -------------------------
# 2) Utility: get best_val_acc per (model, target_name, layer)
# -------------------------
# Your JSON rows include: layer, target_type, target_name, group, train_acc, val_acc, best_val_acc
# We'll use best_val_acc.
def get_curve(df, model, target_name):
    sub = df[(df["model"] == model) & (df["target_name"] == target_name)].copy()
    # Keep only defined layers
    sub = sub[sub["layer"].notna()].sort_values("layer")
    # Some rows might repeat; pick max best_val_acc per layer
    curve = sub.groupby("layer", as_index=False)["best_val_acc"].max()
    curve = curve.set_index("layer").reindex(LAYER_ORDER)["best_val_acc"]
    return curve  # Series indexed by layer order

# -------------------------
# 3) Emergence definition
# -------------------------
# Emergence layer = earliest layer where best_val_acc >= (threshold * max_acc_over_layers)
# (This matches your earlier approach; can change threshold)
def compute_emergence_table(df, target_type="attribute", threshold=0.90):
    sub = df[df["target_type"] == target_type].copy()

    # max acc per (model, target_name)
    max_acc = sub.groupby(["model", "target_name"], as_index=False)["best_val_acc"].max()
    max_acc = max_acc.rename(columns={"best_val_acc": "max_acc"})

    merged = sub.merge(max_acc, on=["model", "target_name"], how="left")
    merged["meets"] = merged["best_val_acc"] >= threshold * merged["max_acc"]

    # earliest layer meeting the threshold
    merged = merged.sort_values(["model", "target_name", "layer"])
    first_meet = merged[merged["meets"]].groupby(["model", "target_name"], as_index=False).first()

    # attach group
    group_map = sub.groupby("target_name", as_index=False)["group"].first()
    first_meet = first_meet.merge(group_map, on="target_name", how="left")

    # if never meets, drop (or set to NaN)
    first_meet["emergence_layer"] = first_meet["layer"]
    first_meet["emergence_idx"] = first_meet["emergence_layer"].map(LAYER_TO_IDX)

    # keep max_acc too
    first_meet = first_meet.merge(max_acc, on=["model", "target_name"], how="left")

    return first_meet[["model", "target_name", "group", "max_acc", "emergence_layer", "emergence_idx"]]

# Compute emergence for attributes + species
attr_emerge = compute_emergence_table(df, target_type="attribute", threshold=0.90)
species_emerge = compute_emergence_table(df, target_type="subclass", threshold=0.90)  # species is subclass in your df
print("attr_emerge:", attr_emerge.shape, "species_emerge:", species_emerge.shape)

# -------------------------
# 4) Composite Plot A: Panel of probe curves
#    (3 attributes + species, baseline vs cbm)
# -------------------------
def plot_curves_panel(attrs, include_species=True):
    # Determine subplot count
    n = len(attrs) + (1 if include_species else 0)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4*nrows), sharex=True, sharey=False)
    axes = np.array(axes).reshape(-1)

    def plot_one(ax, target_name, title_prefix="Attribute"):
        y_base = get_curve(df, "baseline", target_name)
        y_cbm  = get_curve(df, "cbm", target_name)

        x = np.arange(len(LAYER_ORDER))
        ax.plot(x, y_base.values, marker="o", label="baseline")
        ax.plot(x, y_cbm.values,  marker="o", label="cbm")
        ax.set_xticks(x)
        ax.set_xticklabels(LAYER_ORDER, rotation=0)
        ax.set_ylim(0, 1.0)
        ax.grid(True, alpha=0.3)
        ax.set_title(f"{title_prefix} probe accuracy vs depth\n{target_name}")
        ax.set_xlabel("Layer")
        ax.set_ylabel("Probe accuracy (best val)")
        ax.legend()

    i = 0
    for a in attrs:
        plot_one(axes[i], a, "Attribute")
        i += 1

    if include_species:
        plot_one(axes[i], "species", "Species")
        i += 1

    # Turn off unused axes
    for j in range(i, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

# Choose the same attributes you discussed
attrs_to_plot = [
    "has_throat_color::black",   # moved deeper in your table
    "has_back_color::black",     # stable/shallow
    "has_shape::perching-like"   # representative shape attribute
]

# filter to existing
existing_attr = set(df.loc[df["target_type"] == "attribute", "target_name"].unique())
attrs_to_plot = [a for a in attrs_to_plot if a in existing_attr]
print("Curves panel will plot:", attrs_to_plot, "+ species")
plot_curves_panel(attrs_to_plot, include_species=True)

# -------------------------
# 5) Composite Plot B: Emergence histogram overlay (baseline vs cbm)
# -------------------------
def plot_emergence_hist_overlay(attr_emerge):
    fig, ax = plt.subplots(figsize=(9,5))

    for model in ["baseline", "cbm"]:
        sub = attr_emerge[attr_emerge["model"] == model]
        # count by emergence_layer
        counts = sub["emergence_layer"].value_counts().reindex(LAYER_ORDER, fill_value=0)
        x = np.arange(len(LAYER_ORDER))
        # slight offset to overlay bars
        offset = -0.2 if model == "baseline" else 0.2
        ax.bar(x + offset, counts.values, width=0.4, label=model)

    ax.set_xticks(np.arange(len(LAYER_ORDER)))
    ax.set_xticklabels(LAYER_ORDER)
    ax.set_ylabel("# attributes")
    ax.set_title("Attribute Emergence Distribution: baseline vs CBM")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_emergence_hist_overlay(attr_emerge)

# -------------------------
# 6) Composite Plot C: Emergence scatter (baseline vs cbm)
# -------------------------
def plot_emergence_scatter(attr_emerge):
    # keep only attributes present in both models
    base = attr_emerge[attr_emerge["model"] == "baseline"].set_index("target_name")
    cbm  = attr_emerge[attr_emerge["model"] == "cbm"].set_index("target_name")
    common = base.index.intersection(cbm.index)

    merged = pd.DataFrame({
        "baseline_idx": base.loc[common, "emergence_idx"],
        "cbm_idx": cbm.loc[common, "emergence_idx"],
    })

    fig, ax = plt.subplots(figsize=(7,7))
    ax.scatter(merged["baseline_idx"], merged["cbm_idx"], alpha=0.6)

    # diagonal line
    ax.plot([0, len(LAYER_ORDER)-1], [0, len(LAYER_ORDER)-1], linestyle="--")

    ax.set_xticks(range(len(LAYER_ORDER)))
    ax.set_xticklabels(LAYER_ORDER, rotation=0)
    ax.set_yticks(range(len(LAYER_ORDER)))
    ax.set_yticklabels(LAYER_ORDER, rotation=0)

    ax.set_xlabel("Emergence depth (baseline)")
    ax.set_ylabel("Emergence depth (CBM)")
    ax.set_title("Per-attribute emergence depth: baseline vs CBM")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_emergence_scatter(attr_emerge)

# -------------------------
# 7) Composite Plot D: Groups with largest mean depth increase under CBM
# -------------------------
def plot_group_depth_shift(attr_emerge, min_group_size=3, top_k=12):
    base = attr_emerge[attr_emerge["model"]=="baseline"]
    cbm  = attr_emerge[attr_emerge["model"]=="cbm"]

    # mean emergence by group (using emergence_idx)
    base_g = base.groupby("group")["emergence_idx"].mean()
    cbm_g  = cbm.groupby("group")["emergence_idx"].mean()

    # counts by group
    cnt = base.groupby("group").size()

    # common groups and filter small groups
    common_groups = base_g.index.intersection(cbm_g.index)
    common_groups = [g for g in common_groups if cnt.get(g,0) >= min_group_size]

    df_g = pd.DataFrame({
        "baseline_mean": base_g.loc[common_groups],
        "cbm_mean": cbm_g.loc[common_groups],
        "count": cnt.loc[common_groups],
    })
    df_g["delta"] = df_g["cbm_mean"] - df_g["baseline_mean"]
    df_g = df_g.sort_values("delta", ascending=False).head(top_k)

    fig, ax = plt.subplots(figsize=(12,5))
    x = np.arange(len(df_g))
    ax.bar(x - 0.2, df_g["baseline_mean"], width=0.4, label="baseline")
    ax.bar(x + 0.2, df_g["cbm_mean"], width=0.4, label="cbm")
    ax.set_xticks(x)
    ax.set_xticklabels(df_g.index, rotation=30, ha="right")
    ax.set_ylabel("Mean emergence depth (index)")
    ax.set_title(f"Groups with largest increase in mean emergence depth under CBM (n≥{min_group_size})")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_group_depth_shift(attr_emerge, min_group_size=3, top_k=12)

# -------------------------
# 8) Composite Plot E (optional): "Main results" page
#    histogram + scatter + species curve all together
# -------------------------
def plot_main_results_page(attr_emerge):
    fig = plt.figure(figsize=(14,10))

    # (1) histogram
    ax1 = fig.add_subplot(2,2,1)
    for model in ["baseline", "cbm"]:
        sub = attr_emerge[attr_emerge["model"] == model]
        counts = sub["emergence_layer"].value_counts().reindex(LAYER_ORDER, fill_value=0)
        x = np.arange(len(LAYER_ORDER))
        offset = -0.2 if model == "baseline" else 0.2
        ax1.bar(x + offset, counts.values, width=0.4, label=model)
    ax1.set_xticks(np.arange(len(LAYER_ORDER)))
    ax1.set_xticklabels(LAYER_ORDER)
    ax1.set_title("Attribute emergence distribution")
    ax1.set_ylabel("# attributes")
    ax1.legend()
    ax1.grid(True, axis="y", alpha=0.3)

    # (2) scatter
    ax2 = fig.add_subplot(2,2,2)
    base = attr_emerge[attr_emerge["model"] == "baseline"].set_index("target_name")
    cbm  = attr_emerge[attr_emerge["model"] == "cbm"].set_index("target_name")
    common = base.index.intersection(cbm.index)
    ax2.scatter(base.loc[common,"emergence_idx"], cbm.loc[common,"emergence_idx"], alpha=0.6)
    ax2.plot([0, len(LAYER_ORDER)-1], [0, len(LAYER_ORDER)-1], linestyle="--")
    ax2.set_xticks(range(len(LAYER_ORDER))); ax2.set_xticklabels(LAYER_ORDER)
    ax2.set_yticks(range(len(LAYER_ORDER))); ax2.set_yticklabels(LAYER_ORDER)
    ax2.set_xlabel("baseline"); ax2.set_ylabel("cbm")
    ax2.set_title("Per-attribute emergence: baseline vs CBM")
    ax2.grid(True, alpha=0.3)

    # (3) species curve
    ax3 = fig.add_subplot(2,1,2)
    y_base = get_curve(df, "baseline", "species")
    y_cbm  = get_curve(df, "cbm", "species")
    x = np.arange(len(LAYER_ORDER))
    ax3.plot(x, y_base.values, marker="o", label="baseline")
    ax3.plot(x, y_cbm.values, marker="o", label="cbm")
    ax3.set_xticks(x); ax3.set_xticklabels(LAYER_ORDER)
    ax3.set_ylim(0,1.0)
    ax3.set_title("Species probe accuracy vs depth (baseline vs CBM)")
    ax3.set_xlabel("Layer")
    ax3.set_ylabel("Probe accuracy (best val)")
    ax3.grid(True, alpha=0.3)
    ax3.legend()

    plt.tight_layout()
    plt.show()

plot_main_results_page(attr_emerge)
